# Three Ways to Call the Same Tool

One tool (`get_weather`), three approaches.  
Each section uses **exactly the same tool function** — only the wiring around it changes.

```
Approach A — Manual          : function + schema + if/elif
Approach B — Dispatcher      : function + schema + dict lookup
Approach C — @beta_tool      : decorated function only (SDK does the rest)
```

Read them top to bottom to see what each approach saves you from writing.

---
## 0 — Setup (shared by all three approaches)

In [ ]:
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"   # cheap model — fine for learning

# ── The ONE tool function — identical in all three approaches ─────────────────
def get_weather(city: str) -> str:
    """Return simulated weather for a city."""
    data = {
        "london":   "12°C, overcast",
        "paris":    "18°C, sunny",
        "tokyo":    "22°C, humid",
        "new york": "15°C, cloudy",
    }
    return data.get(city.lower(), f"No data for {city!r}")

# Quick sanity check
print(get_weather("paris"))   # 18°C, sunny

---
## Approach A — Manual (what 001_tools.ipynb showed)

**You write:**
1. The Python function  
2. The JSON schema (by hand)  
3. The `if/elif` block that matches Claude's requested tool name to your function  
4. The follow-up API call to send the result back  

This is the most explicit approach — nothing is hidden.

In [ ]:
# ── A1. Write the schema by hand ──────────────────────────────────────────────
# Claude never sees your Python function.
# It only sees this JSON description to know the tool exists.

weather_schema = {
    "name": "get_weather",
    "description": "Return the current weather for a given city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city name, e.g. London or Tokyo.",
            }
        },
        "required": ["city"],
    },
}

print("Schema written. Claude will read this to know the tool exists.")

In [ ]:
# ── A2. Call 1 — send the user message + schema to Claude ─────────────────────
# Claude reads the schema and decides to use the tool.

messages = [{"role": "user", "content": "What's the weather in Paris?"}]

response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    tools=[weather_schema],   # ← pass the schema here
    messages=messages,
)

print("stop_reason :", response.stop_reason)  # should be 'tool_use'
print("content     :", response.content)

In [ ]:
# ── A3. Extract what Claude is requesting ──────────────────────────────────────
# Claude didn't call anything. It wrote a tool_use block saying
# "please call get_weather with city='Paris'".

tool_block = next(b for b in response.content if b.type == "tool_use")

print("Tool Claude wants:", tool_block.name)    # "get_weather"
print("Arguments        :", tool_block.input)   # {"city": "Paris"}
print("Request ID       :", tool_block.id)      # "toolu_..." — you need this later

In [ ]:
# ── A4. YOU call the Python function (the if/elif dispatcher) ──────────────────
# Claude gave you a NAME (string) and ARGUMENTS (dict).
# You have to match that name to an actual function yourself.

if tool_block.name == "get_weather":
    tool_result = get_weather(**tool_block.input)          # unpack the dict as kwargs
elif tool_block.name == "some_other_tool":
    tool_result = "..."
else:
    tool_result = f"Error: unknown tool '{tool_block.name}'"

print("Result:", tool_result)

In [ ]:
# ── A5. Call 2 — send the result back so Claude can answer ────────────────────
# Must include: the assistant turn (with the tool_use block) AND
# a new user turn (with the tool_result block).

messages.append({"role": "assistant", "content": response.content})
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,   # ties this result to the request above
        "content": tool_result,
    }],
})

final = client.messages.create(
    model=MODEL,
    max_tokens=256,
    tools=[weather_schema],
    messages=messages,
)

print("Final answer:")
print(final.content[0].text)

### What Approach A requires you to write

```
✍️  get_weather()          — the Python function
✍️  weather_schema         — the JSON schema (by hand)
✍️  if tool_block.name == ... — the dispatcher (by hand)
✍️  Call 1 + Call 2        — both API calls, wired manually
```

Total boilerplate: **schema + dispatcher + 2 API calls**.

---
## Approach B — Dispatcher Dictionary

Same as Approach A but replaces the `if/elif` chain with a dictionary lookup.  
Scales to many tools without getting messy.

**What changes:** only the dispatcher (step A4).  Everything else is identical.

In [ ]:
# ── B1. Schema — same as before (still required) ──────────────────────────────
weather_schema = {
    "name": "get_weather",
    "description": "Return the current weather for a given city.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name"}
        },
        "required": ["city"],
    },
}

In [ ]:
# ── B2. The dispatcher — a dict instead of if/elif ────────────────────────────
# Each value is a lambda that accepts the `input` dict Claude provides.
#
# Why lambda?  Claude passes ONE dict (tool_block.input).
# The lambda's job is to unpack that dict and call the real function.

TOOL_REGISTRY = {
    #  name Claude uses       how to call the real function
    #  ─────────────────────  ────────────────────────────────────────
    "get_weather":            lambda inp: get_weather(inp["city"]),
    # add more tools here as your project grows:
    # "calculate":            lambda inp: calculate(inp["expression"]),
    # "get_current_time":     lambda inp: get_current_time(),
}

print("Dispatcher registered:", list(TOOL_REGISTRY))

In [ ]:
# ── B3. Call 1 — same as A2 ───────────────────────────────────────────────────
messages = [{"role": "user", "content": "What's the weather in Tokyo?"}]

response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    tools=[weather_schema],
    messages=messages,
)
tool_block = next(b for b in response.content if b.type == "tool_use")

print("Tool:", tool_block.name, "| Args:", tool_block.input)

In [ ]:
# ── B4. Dispatch — one line instead of if/elif ────────────────────────────────
# TOOL_REGISTRY[name]  →  looks up the lambda
# (tool_block.input)   →  calls it with Claude's arguments

fn          = TOOL_REGISTRY[tool_block.name]  # step 1: look up
tool_result = fn(tool_block.input)            # step 2: call

# Or in one line:
# tool_result = TOOL_REGISTRY[tool_block.name](tool_block.input)

print("Result:", tool_result)

In [ ]:
# ── B5. Call 2 — same as A5 ───────────────────────────────────────────────────
messages.append({"role": "assistant", "content": response.content})
messages.append({
    "role": "user",
    "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": tool_result}],
})

final = client.messages.create(
    model=MODEL, max_tokens=256, tools=[weather_schema], messages=messages
)
print("Final answer:")
print(final.content[0].text)

### What Approach B requires you to write

```
✍️  get_weather()          — the Python function  (same)
✍️  weather_schema         — the JSON schema      (same)
✍️  TOOL_REGISTRY          — dict instead of if/elif  ← improvement
✍️  Call 1 + Call 2        — both API calls       (same)
```

The only improvement over A is the dispatcher is cleaner.  
You still write the schema by hand and manage both API calls.

---
## Approach C — `@beta_tool` + Tool Runner

The SDK generates the schema from your type hints and docstring.  
The tool runner manages the loop (Call 1 → dispatch → Call 2 → repeat) automatically.

**What changes:** almost everything is gone.

In [ ]:
from anthropic import beta_tool

# ── C1. Decorate the function — no separate schema needed ─────────────────────
# The decorator reads:
#   • city: str       → property named "city" with type "string"
#   • "Get the current weather..."  → description
#   • Args: city: ...  → property description

@beta_tool
def get_weather_tool(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The city name, e.g. London or Tokyo.
    """
    return get_weather(city)   # ← calls our original function


# Peek at the auto-generated schema to see what the decorator produced
import json
print("Auto-generated schema:")
print(json.dumps(get_weather_tool.tool_schema, indent=2))

In [ ]:
# ── C2. Hand everything to the tool runner — it loops until done ──────────────
# No Call 1 / Call 2 split.
# No dispatcher.
# No extracting tool_block manually.
# The SDK does all of that internally.

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=256,
    tools=[get_weather_tool],   # pass the decorated function directly
    messages=[{"role": "user", "content": "What's the weather in London?"}],
)

for message in runner:
    print(f"stop_reason={message.stop_reason}")
    for block in message.content:
        if block.type == "text":
            print("Final answer:", block.text)

### What Approach C requires you to write

```
✍️  @beta_tool get_weather_tool()  — decorated function (schema auto-generated)
✍️  tool_runner(...)               — one call, loop handled internally

❌  weather_schema                 — not needed (auto-generated)
❌  TOOL_REGISTRY / if/elif        — not needed (SDK dispatches for you)
❌  Call 2 (tool_result turn)      — not needed (SDK sends it for you)
```

---
## Side-by-side comparison

Same question (`"What's the weather in London?"`) across all three approaches.

In [ ]:
# ── Helper: run all three and compare ────────────────────────────────────────

QUESTION = "What's the weather in New York?"

# ── Approach A ────────────────────────────────────────────────────────────────
msgs_a = [{"role": "user", "content": QUESTION}]
r1 = client.messages.create(model=MODEL, max_tokens=256, tools=[weather_schema], messages=msgs_a)
tb = next(b for b in r1.content if b.type == "tool_use")
result_a_raw = get_weather(**tb.input)                   # manual if/elif (here: direct call)
msgs_a.append({"role": "assistant", "content": r1.content})
msgs_a.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tb.id, "content": result_a_raw}]})
r2 = client.messages.create(model=MODEL, max_tokens=256, tools=[weather_schema], messages=msgs_a)
answer_a = r2.content[0].text

# ── Approach B ────────────────────────────────────────────────────────────────
msgs_b = [{"role": "user", "content": QUESTION}]
r1 = client.messages.create(model=MODEL, max_tokens=256, tools=[weather_schema], messages=msgs_b)
tb = next(b for b in r1.content if b.type == "tool_use")
result_b_raw = TOOL_REGISTRY[tb.name](tb.input)          # dict dispatcher
msgs_b.append({"role": "assistant", "content": r1.content})
msgs_b.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tb.id, "content": result_b_raw}]})
r2 = client.messages.create(model=MODEL, max_tokens=256, tools=[weather_schema], messages=msgs_b)
answer_b = r2.content[0].text

# ── Approach C ────────────────────────────────────────────────────────────────
runner = client.beta.messages.tool_runner(
    model=MODEL, max_tokens=256,
    tools=[get_weather_tool],
    messages=[{"role": "user", "content": QUESTION}],
)
answer_c = ""
for msg in runner:
    for block in msg.content:
        if block.type == "text":
            answer_c = block.text

# ── Print all three ───────────────────────────────────────────────────────────
print(f"Question : {QUESTION}")
print()
print(f"A (manual)     : {answer_a}")
print(f"B (dispatcher) : {answer_b}")
print(f"C (beta_tool)  : {answer_c}")
print()
print("All three produce the same answer — only the wiring differs.")

---
## Key mental model — one diagram

```
WHAT YOU WRITE                       WHAT THE SDK / YOU DOES
──────────────────────────────────   ────────────────────────────────────────────

Approach A
  Python function          ──────►  YOU call it inside if/elif
  JSON schema (by hand)    ──────►  YOU pass to Claude
  if/elif dispatcher       ──────►  YOU match name → function
  2 API calls              ──────►  YOU wire manually

Approach B
  Python function          ──────►  YOU call it via dict lookup
  JSON schema (by hand)    ──────►  YOU pass to Claude
  dict TOOL_REGISTRY       ──────►  YOU match name → function  (cleaner)
  2 API calls              ──────►  YOU wire manually

Approach C
  @beta_tool function      ──────►  SDK generates schema from hints + docstring
                                    SDK dispatches to your function
                                    SDK handles both API calls and the loop
```

### When to use each

| Situation | Use |
|---|---|
| Learning / understanding the protocol | **A — manual** |
| Many tools, need custom logging or approval gates | **B — dispatcher** |
| Production code, want minimal boilerplate | **C — `@beta_tool`** |